# 📊 Uplift Modeling para Retenção de Clientes — v2.0
**Melhorias:** +5 features de negócio | +Naive Bayes | +Precision/Recall/KS

**Referências:**
1. Gutierrez, P., & Gérardy, J. Y. (2017). Causal Inference and Uplift Modelling. JMLR.
2. Radcliffe, N. J. (2007). Using control groups to target on predicted lift. DMQ.
3. Scikit-Learn Docs — https://scikit-learn.org
4. Plotly Docs — https://plotly.com/python/
5. Pandas Docs — https://pandas.pydata.org/docs/


## Etapa 1 — EDA e Preparação

In [ ]:
# Importações de todas as bibliotecas usadas no projeto
# Referência: PEP8 Standard Imports
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, accuracy_score, f1_score,
                              precision_score, recall_score)
from sklearn.inspection import permutation_importance
from scipy.stats import ks_2samp

import warnings
warnings.filterwarnings('ignore')

import plotly.io as pio
pio.renderers.default = "notebook_connected"

print("✅ Bibliotecas carregadas!")

In [ ]:
# Carregamento dos 3 datasets isoladamente
# Justificativa: Clientes=Quem, Campanha=Ação, Resposta=Target
# Referência: pd.read_csv — https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html

CAMINHO = 'Datasets_Projeto_Pratico_DSCS/'
df_clientes = pd.read_csv(f'{CAMINHO}clientes.csv')
df_campanha = pd.read_csv(f'{CAMINHO}campanha.csv')
df_resposta = pd.read_csv(f'{CAMINHO}resposta.csv')

print(f"Clientes: {df_clientes.shape} | Campanha: {df_campanha.shape} | Resposta: {df_resposta.shape}")

In [ ]:
# Análise de Missing Values — Cada Dataset Individualmente
# Justificativa de Negócio: Dados ausentes indicam falhas no CRM da empresa.
# Analisamos ANTES do join para não contaminar a base final.
# Referência: isnull() — https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.isnull.html

def analisar_missing(df, nome):
    qtd = df.isnull().sum()
    pct = (qtd / len(df) * 100).round(2)
    resumo = pd.DataFrame({'Quantidade': qtd, 'Percentual(%)': pct})
    print(f"\n{'='*50}")
    print(f" Dataset: {nome} — {len(df):,} registros | {len(df.columns)} colunas")
    print(f"{'='*50}")
    print(resumo.to_string())
    if qtd.sum() == 0:
        print("  ✅ Sem dados ausentes!")
    else:
        print(f"  ⚠️ {qtd.sum()} células ausentes!")

analisar_missing(df_clientes, 'clientes.csv')
analisar_missing(df_campanha, 'campanha.csv')
analisar_missing(df_resposta, 'resposta.csv')

In [ ]:
# Análise Univariada — Histogramas das variáveis numéricas
# Justificativa: Entender se as distribuições são simétricas, enviesadas ou bimodais
# impacta a escolha de algoritmos (ex: Naive Bayes assume distribuição normal).
# Referência: Plotly Subplots — https://plotly.com/python/subplots/

colunas_num = ['idade', 'renda_mensal', 'tempo_como_cliente (meses)', 'score_satisfacao']
fig = make_subplots(rows=2, cols=2,
    subplot_titles=['Distribuição Idade', 'Distribuição Renda Mensal',
                    'Tempo como Cliente (meses)', 'Score de Satisfação'])
cores = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA']
for i, col in enumerate(colunas_num):
    fig.add_trace(go.Histogram(x=df_clientes[col], name=col,
                               marker_color=cores[i], nbinsx=30), row=i//2+1, col=i%2+1)
fig.update_layout(title_text='📊 Análise Univariada — Perfis dos Clientes',
                  template='plotly_white', showlegend=False, height=600)
fig.show()

In [ ]:
# Análise Bivariada — Renda vs Idade com Gênero e Satisfação
# Justificativa: Marketing cria réguas por ciclo de vida e poder aquisitivo.
# Referência: Plotly Scatter — https://plotly.com/python/line-and-scatter/

fig2 = px.scatter(df_clientes, x='idade', y='renda_mensal', color='genero',
    size='score_satisfacao',
    title='🔍 Bivariada: Renda vs Idade (tamanho = Satisfação | cor = Gênero)',
    labels={'idade':'Idade (anos)', 'renda_mensal':'Renda (R$)', 'genero':'Gênero'},
    template='plotly_white', opacity=0.7)
fig2.show()

In [ ]:
# Análise Multivariada — Mapa de Correlação (Pearson)
# Justificativa: Detectar multicolinearidade antes do treino de modelos lineares como
# Regressão Logística e Naive Bayes, que são sensíveis a variáveis redundantes.
# Referência: df.corr() — https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corr.html

numericas = df_clientes.select_dtypes(include='number')
corr = numericas.corr().round(2)
fig3 = px.imshow(corr, text_auto=True,
    title='🔗 Multivariada — Mapa de Correlação (Pearson)',
    color_continuous_scale='RdBu_r', template='plotly_white')
fig3.show()

In [ ]:
# Merge das 3 tabelas → Base Analítica Única (ABT)
# Justificativa: Cada cliente precisa ter: perfil + ação recebida + resultado obtido
# grupo=1 Tratamento (recebeu campanha), grupo=0 Controle (não recebeu)
# Referência: pd.merge — https://pandas.pydata.org/docs/user_guide/merging.html

df_base = pd.merge(df_clientes, df_campanha, on='id_cliente', how='inner')
df_base = pd.merge(df_base, df_resposta, on='id_cliente', how='inner')
df_base.rename(columns={'recebeu_campanha': 'grupo', 'manteve_contrato': 'target'}, inplace=True)

print(f"✅ ABT criada: {df_base.shape[0]:,} registros | {df_base.shape[1]} colunas")
print(f"\n Distribuição Grupos:")
print(df_base['grupo'].value_counts().rename({0:'Controle (0)', 1:'Tratamento (1)'}))
df_base.head()

In [ ]:
# Teste de Balanceamento do Experimento A/B
# Justificativa: Valida que o envio da campanha foi aleatório (sem viés de renda/idade).
# Se grupos forem desbalanceados, o uplift será superestimado ou subestimado.
# Referência: Plotly Box — https://plotly.com/python/box-plots/

taxa = df_base.groupby('grupo')['target'].mean().reset_index()
uplift_bruto = taxa[taxa['grupo']==1]['target'].values[0] - taxa[taxa['grupo']==0]['target'].values[0]
print(f"📈 Uplift Médio Observado (ATE): {uplift_bruto:.4f} ({uplift_bruto*100:.2f}%)")

fig4 = px.box(df_base, x='grupo', y='renda_mensal', color='grupo',
    title='⚖️ Balanceamento: Renda Mensal por Grupo (Tratamento vs Controle)',
    labels={'grupo':'Grupo (0=Controle, 1=Tratamento)', 'renda_mensal':'Renda (R$)'},
    template='plotly_white', color_discrete_map={0:'#636EFA', 1:'#EF553B'})
fig4.update_layout(showlegend=False)
fig4.show()

---
## ⚙️ Etapa 2 — Engenharia de Atributos e Pré-Processamento
Criamos **7 features** com justificativas de negócio claras.

In [ ]:
# Feature 1 — genero_num: Conversão numérica do gênero
# Justificativa: Modelos como Naive Bayes e Regressão Logística precisam de entradas numéricas.
# M=1, F=0 (label encoding simples para variável binária sem hierarquia implícita).
# Referência: Label Encoding — https://scikit-learn.org/stable/modules/preprocessing.html

df_base['genero_num'] = df_base['genero'].map({'M': 1, 'F': 0})

contagem = df_base['genero_num'].value_counts()
print(f"Feature 1 — genero_num:")
print(f"  M=1: {contagem.get(1,0)} clientes | F=0: {contagem.get(0,0)} clientes")

In [ ]:
# Feature 2 — renda_por_idade: Renda média por ano de vida (proxy de produtividade financeira)
# Justificativa: Um cliente de 30 anos com R$9.000 tem perfil financeiro muito diferente
# de um de 60 anos com a mesma renda. Essa feature captura a trajetória de acúmulo de riqueza.
# Referência: Feature Engineering for ML — Zheng & Casari (2018), O'Reilly.

df_base['renda_por_idade'] = (df_base['renda_mensal'] / df_base['idade']).round(2)

print(f"Feature 2 — renda_por_idade:")
print(f"  Média: R${df_base['renda_por_idade'].mean():.2f} por ano de vida")
print(f"  Min: R${df_base['renda_por_idade'].min():.2f} | Max: R${df_base['renda_por_idade'].max():.2f}")

In [ ]:
# Feature 3 — score_fidelidade: Combina tempo de relacionamento com satisfação
# Justificativa: Clientes com longa permanência E alta satisfação são os mais valiosos.
# Essa feature cria um índice composto de "fidelidade real" do cliente.
# Referência: CLV (Customer Lifetime Value) concept — Kumar & Reinartz (2016), Springer.

df_base['score_fidelidade'] = (
    df_base['tempo_como_cliente (meses)'] * df_base['score_satisfacao']
).round(2)

print(f"Feature 3 — score_fidelidade:")
print(f"  Média: {df_base['score_fidelidade'].mean():.1f}")
print(f"  Min: {df_base['score_fidelidade'].min()} | Max: {df_base['score_fidelidade'].max()}")

In [ ]:
# Feature 4 — renda_alta: Flag binária para clientes premium (acima da mediana de renda)
# Justificativa: Campanhas de retenção têm custo (desconto, brinde). Para clientes de renda alta,
# o ROI é maior pois a receita mensal também é maior. Segmentar por renda orienta o investimento.
# Referência: RFM Analysis — Blattberg, Kim & Neslin (2008), Springer.

mediana_renda = df_base['renda_mensal'].median()
df_base['renda_alta'] = (df_base['renda_mensal'] >= mediana_renda).astype(int)

qtd_premium = df_base['renda_alta'].sum()
print(f"Feature 4 — renda_alta (corte na mediana = R${mediana_renda:,.2f}):")
print(f"  {qtd_premium} clientes premium ({qtd_premium/len(df_base)*100:.1f}%)")

In [ ]:
# Feature 5 — cliente_antigo: Flag para clientes com mais de 12 meses (1 ano) de relacionamento
# Justificativa: Clientes com mais de 1 ano de permanência têm menor custo de retenção e
# maior probabilidade de resposta positiva a campanhas (efeito de ancoragem comportamental).
# Referência: Behavioral Economics — Thaler & Sunstein (2008), Nudge, Penguin Books.

df_base['cliente_antigo'] = (df_base['tempo_como_cliente (meses)'] > 12).astype(int)

qtd_antigos = df_base['cliente_antigo'].sum()
print(f"Feature 5 — cliente_antigo (> 12 meses):")
print(f"  {qtd_antigos} clientes antigos ({qtd_antigos/len(df_base)*100:.1f}%)")

In [ ]:
# Feature 6 — faixa_etaria: Categorização por ciclo de vida
# Justificativa: Cada geração tem perfil de consumo e resposta a campanhas distinto.
# Jovens respondem melhor a canais digitais; Sêniors preferem contato humanizado.
# Referência: Generational Marketing — Kotler & Keller (2016), Marketing Management.

def classificar_faixa(idade):
    if idade < 30:   return 'Jovem'
    elif idade < 45: return 'Adulto'
    elif idade < 60: return 'Maduro'
    else:            return 'Senior'

df_base['faixa_etaria'] = df_base['idade'].apply(classificar_faixa)
print("Feature 6 — faixa_etaria:")
print(df_base['faixa_etaria'].value_counts())

In [ ]:
# Feature 7 — score_valor_cliente: Índice composto CLV Proxy (0 a 1)
# Justificativa: Combina renda, tempo de relacionamento e satisfação em um único score normalizado.
# Usado para priorizar clientes em cenários de budget limitado de marketing.
# Referência: CLV Scoring — Fader, Hardie & Lee (2005), Counting Your Customers the Easy Way.

def normalizar(serie):
    # Normalização Min-Max simples entre 0 e 1
    return (serie - serie.min()) / (serie.max() - serie.min())

df_base['score_valor_cliente'] = (
    0.4 * normalizar(df_base['renda_mensal']) +
    0.3 * normalizar(df_base['tempo_como_cliente (meses)']) +
    0.3 * normalizar(df_base['score_satisfacao'])
).round(4)

print(f"Feature 7 — score_valor_cliente (0 a 1):")
print(df_base['score_valor_cliente'].describe().round(3))

In [ ]:
# Visualização das Novas Features — Radar de Perfil por Grupo
# Justificativa: Visualizar como as features se distribuem entre Tratamento e Controle
# confirma se as features criadas são discriminantes para o modelo.

comparativo = df_base.groupby('grupo')[
    ['renda_por_idade','score_fidelidade','score_valor_cliente','renda_alta','cliente_antigo']
].mean().round(3).reset_index()
comparativo['grupo'] = comparativo['grupo'].map({0:'Controle', 1:'Tratamento'})

fig5 = px.bar(comparativo.melt(id_vars='grupo'), x='variable', y='value', color='grupo',
    barmode='group',
    title='📊 Perfil Médio das Novas Features por Grupo (Tratamento vs Controle)',
    labels={'variable':'Feature', 'value':'Valor Médio', 'grupo':'Grupo'},
    template='plotly_white')
fig5.show()

In [ ]:
# Pré-Processamento: One-Hot Encoding e remoção de colunas não preditivas
# Justificativa: Algoritmos de ML precisam de entradas numéricas. id_cliente não é sinal preditivo.
# Referência: One-Hot Encoding — https://pandas.pydata.org/docs/reference/api/pandas.get_dummies.html

df_model = pd.get_dummies(df_base, columns=['genero', 'faixa_etaria'], drop_first=True)
df_model.drop('id_cliente', axis=1, inplace=True)

print("✅ Tipo de dados da base analítica:")
print(df_model.dtypes)
print(f"\nDimensão final: {df_model.shape}")

In [ ]:
# Divisão Treino/Teste e Separação das Variáveis de Controle
# X=features | y=target (reteve?) | w=grupo (recebeu campanha?)
# random_state=42 garante reprodutibilidade — mesmo resultado toda vez que rodar.
# Referência: train_test_split — https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html

X = df_model.drop(['grupo', 'target'], axis=1)
y = df_model['target']
w = df_model['grupo']

X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X, y, w, test_size=0.30, random_state=42, stratify=y)

print(f"✅ Split: Treino={X_train.shape[0]:,} | Teste={X_test.shape[0]:,} | Features={X_train.shape[1]}")

---
## 🤖 Etapa 3 — Machine Learning: T-Learner com 4 Algoritmos

| # | Família | Algoritmo | Nível |
|---|---|---|---|
| 1 | Probabilístico Bayesiano | **Naive Bayes** | Baseline |
| 2 | Probabilístico Linear | **Regressão Logística** | Básico |
| 3 | Ensemble Bagging | **Random Forest** | Intermediário |
| 4 | Ensemble Boosting | **Gradient Boosting** | Avançado |

**T-Learner:** Dois modelos paralelos (M_controle e M_tratamento). Uplift = P(Y|T=1) - P(Y|T=0)

Referência: Künzel et al. (2019). Metalearners for estimating heterogeneous treatment effects. PNAS.


In [ ]:
# Funções do T-Learner (sem classes, nível júnior)
# Referência: Metalearners — Künzel et al. (2019) PNAS

def treinar_t_learner(classe_modelo, kwargs_modelo, X_tr, y_tr, w_tr):
    # Instancia dois modelos separados: um para cada grupo do experimento
    m_ctrl = classe_modelo(**kwargs_modelo)
    m_trat = classe_modelo(**kwargs_modelo)
    # Treina cada modelo apenas com os dados do seu respectivo grupo
    m_ctrl.fit(X_tr[w_tr==0], y_tr[w_tr==0])
    m_trat.fit(X_tr[w_tr==1], y_tr[w_tr==1])
    return m_ctrl, m_trat

def calcular_uplift(m_ctrl, m_trat, X_novo):
    # Uplift = Probabilidade no mundo "Com Campanha" - Probabilidade no mundo "Sem Campanha"
    return m_trat.predict_proba(X_novo)[:,1] - m_ctrl.predict_proba(X_novo)[:,1]

def calcular_ks(y_true, y_proba):
    # KS (Kolmogorov-Smirnov): mede a separação máxima entre distribuições de positivos e negativos
    # Quanto maior o KS, melhor o modelo discrimina entre quem retém e quem cancela
    pos = y_proba[y_true == 1]
    neg = y_proba[y_true == 0]
    ks_stat, _ = ks_2samp(pos, neg)
    return ks_stat

print("✅ Funções T-Learner, Uplift e KS definidas!")

In [ ]:
# Modelo 1 — Naive Bayes (Probabilístico Bayesiano — Baseline)
# Justificativa: Naive Bayes assume independência entre as features e usa o Teorema de Bayes.
# É o modelo mais simples e rápido — serve como piso de performance (baseline).
# Se os outros modelos não baterem o NB, algo está errado no processo.
# Referência: Naive Bayes — Murphy, K. P. (2012). Machine Learning: A Probabilistic Perspective. MIT Press.

print("== MODELO 1: Naive Bayes (Probabilístico Bayesiano - Baseline) ==")

m_ctrl_nb, m_trat_nb = treinar_t_learner(GaussianNB, {}, X_train, y_train, w_train)
uplift_nb = calcular_uplift(m_ctrl_nb, m_trat_nb, X_test)

# Métricas Globais
auc_ctrl_nb  = roc_auc_score(y_test[w_test==0], m_ctrl_nb.predict_proba(X_test[w_test==0])[:,1])
auc_trat_nb  = roc_auc_score(y_test[w_test==1], m_trat_nb.predict_proba(X_test[w_test==1])[:,1])
acc_ctrl_nb  = accuracy_score(y_test[w_test==0], m_ctrl_nb.predict(X_test[w_test==0]))
acc_trat_nb  = accuracy_score(y_test[w_test==1], m_trat_nb.predict(X_test[w_test==1]))
f1_ctrl_nb   = f1_score(y_test[w_test==0], m_ctrl_nb.predict(X_test[w_test==0]))
f1_trat_nb   = f1_score(y_test[w_test==1], m_trat_nb.predict(X_test[w_test==1]))
prec_ctrl_nb = precision_score(y_test[w_test==0], m_ctrl_nb.predict(X_test[w_test==0]))
prec_trat_nb = precision_score(y_test[w_test==1], m_trat_nb.predict(X_test[w_test==1]))
rec_ctrl_nb  = recall_score(y_test[w_test==0], m_ctrl_nb.predict(X_test[w_test==0]))
rec_trat_nb  = recall_score(y_test[w_test==1], m_trat_nb.predict(X_test[w_test==1]))
ks_ctrl_nb   = calcular_ks(y_test[w_test==0].values, m_ctrl_nb.predict_proba(X_test[w_test==0])[:,1])
ks_trat_nb   = calcular_ks(y_test[w_test==1].values, m_trat_nb.predict_proba(X_test[w_test==1])[:,1])

print(f"  AUC       Controle: {auc_ctrl_nb:.4f} | AUC       Tratamento: {auc_trat_nb:.4f}")
print(f"  Accuracy  Controle: {acc_ctrl_nb:.4f} | Accuracy  Tratamento: {acc_trat_nb:.4f}")
print(f"  F1-Score  Controle: {f1_ctrl_nb:.4f} | F1-Score  Tratamento: {f1_trat_nb:.4f}")
print(f"  Precision Controle: {prec_ctrl_nb:.4f} | Precision Tratamento: {prec_trat_nb:.4f}")
print(f"  Recall    Controle: {rec_ctrl_nb:.4f} | Recall    Tratamento: {rec_trat_nb:.4f}")
print(f"  KS        Controle: {ks_ctrl_nb:.4f} | KS        Tratamento: {ks_trat_nb:.4f}")
print(f"  Uplift Médio (Teste): {uplift_nb.mean():.4f}")

In [ ]:
# Modelo 2 — Regressão Logística (Probabilístico Linear — Básico)
# Justificativa: Modela a probabilidade de retenção como função linear das features.
# Mais interpretável que modelos complexos — permite ver os coeficientes de cada variável.
# Referência: Hosmer, D. W., & Lemeshow, S. (2000). Applied Logistic Regression. Wiley.

print("== MODELO 2: Regressão Logística (Probabilístico Linear) ==")

m_ctrl_lr, m_trat_lr = treinar_t_learner(
    LogisticRegression, {'max_iter': 1000, 'random_state': 42}, X_train, y_train, w_train)
uplift_lr = calcular_uplift(m_ctrl_lr, m_trat_lr, X_test)

auc_ctrl_lr  = roc_auc_score(y_test[w_test==0], m_ctrl_lr.predict_proba(X_test[w_test==0])[:,1])
auc_trat_lr  = roc_auc_score(y_test[w_test==1], m_trat_lr.predict_proba(X_test[w_test==1])[:,1])
acc_ctrl_lr  = accuracy_score(y_test[w_test==0], m_ctrl_lr.predict(X_test[w_test==0]))
acc_trat_lr  = accuracy_score(y_test[w_test==1], m_trat_lr.predict(X_test[w_test==1]))
f1_ctrl_lr   = f1_score(y_test[w_test==0], m_ctrl_lr.predict(X_test[w_test==0]))
f1_trat_lr   = f1_score(y_test[w_test==1], m_trat_lr.predict(X_test[w_test==1]))
prec_ctrl_lr = precision_score(y_test[w_test==0], m_ctrl_lr.predict(X_test[w_test==0]))
prec_trat_lr = precision_score(y_test[w_test==1], m_trat_lr.predict(X_test[w_test==1]))
rec_ctrl_lr  = recall_score(y_test[w_test==0], m_ctrl_lr.predict(X_test[w_test==0]))
rec_trat_lr  = recall_score(y_test[w_test==1], m_trat_lr.predict(X_test[w_test==1]))
ks_ctrl_lr   = calcular_ks(y_test[w_test==0].values, m_ctrl_lr.predict_proba(X_test[w_test==0])[:,1])
ks_trat_lr   = calcular_ks(y_test[w_test==1].values, m_trat_lr.predict_proba(X_test[w_test==1])[:,1])

print(f"  AUC       Controle: {auc_ctrl_lr:.4f} | AUC       Tratamento: {auc_trat_lr:.4f}")
print(f"  Accuracy  Controle: {acc_ctrl_lr:.4f} | Accuracy  Tratamento: {acc_trat_lr:.4f}")
print(f"  F1-Score  Controle: {f1_ctrl_lr:.4f} | F1-Score  Tratamento: {f1_trat_lr:.4f}")
print(f"  Precision Controle: {prec_ctrl_lr:.4f} | Precision Tratamento: {prec_trat_lr:.4f}")
print(f"  Recall    Controle: {rec_ctrl_lr:.4f} | Recall    Tratamento: {rec_trat_lr:.4f}")
print(f"  KS        Controle: {ks_ctrl_lr:.4f} | KS        Tratamento: {ks_trat_lr:.4f}")
print(f"  Uplift Médio (Teste): {uplift_lr.mean():.4f}")

In [ ]:
# Modelo 3 — Random Forest (Ensemble Bagging — Intermediário)
# Justificativa: Cria múltiplas árvores em amostras aleatórias e faz votação (bagging).
# Captura não-linearidades e interações entre features melhor que modelos lineares.
# Robusto a outliers e não precisa de normalização das features.
# Referência: Breiman, L. (2001). Random Forests. Machine Learning, 45(1), 5-32.

print("== MODELO 3: Random Forest (Ensemble — Bagging) ==")

m_ctrl_rf, m_trat_rf = treinar_t_learner(
    RandomForestClassifier,
    {'n_estimators': 200, 'max_depth': 8, 'random_state': 42, 'n_jobs': -1},
    X_train, y_train, w_train)
uplift_rf = calcular_uplift(m_ctrl_rf, m_trat_rf, X_test)

auc_ctrl_rf  = roc_auc_score(y_test[w_test==0], m_ctrl_rf.predict_proba(X_test[w_test==0])[:,1])
auc_trat_rf  = roc_auc_score(y_test[w_test==1], m_trat_rf.predict_proba(X_test[w_test==1])[:,1])
acc_ctrl_rf  = accuracy_score(y_test[w_test==0], m_ctrl_rf.predict(X_test[w_test==0]))
acc_trat_rf  = accuracy_score(y_test[w_test==1], m_trat_rf.predict(X_test[w_test==1]))
f1_ctrl_rf   = f1_score(y_test[w_test==0], m_ctrl_rf.predict(X_test[w_test==0]))
f1_trat_rf   = f1_score(y_test[w_test==1], m_trat_rf.predict(X_test[w_test==1]))
prec_ctrl_rf = precision_score(y_test[w_test==0], m_ctrl_rf.predict(X_test[w_test==0]))
prec_trat_rf = precision_score(y_test[w_test==1], m_trat_rf.predict(X_test[w_test==1]))
rec_ctrl_rf  = recall_score(y_test[w_test==0], m_ctrl_rf.predict(X_test[w_test==0]))
rec_trat_rf  = recall_score(y_test[w_test==1], m_trat_rf.predict(X_test[w_test==1]))
ks_ctrl_rf   = calcular_ks(y_test[w_test==0].values, m_ctrl_rf.predict_proba(X_test[w_test==0])[:,1])
ks_trat_rf   = calcular_ks(y_test[w_test==1].values, m_trat_rf.predict_proba(X_test[w_test==1])[:,1])

print(f"  AUC       Controle: {auc_ctrl_rf:.4f} | AUC       Tratamento: {auc_trat_rf:.4f}")
print(f"  Accuracy  Controle: {acc_ctrl_rf:.4f} | Accuracy  Tratamento: {acc_trat_rf:.4f}")
print(f"  F1-Score  Controle: {f1_ctrl_rf:.4f} | F1-Score  Tratamento: {f1_trat_rf:.4f}")
print(f"  Precision Controle: {prec_ctrl_rf:.4f} | Precision Tratamento: {prec_trat_rf:.4f}")
print(f"  Recall    Controle: {rec_ctrl_rf:.4f} | Recall    Tratamento: {rec_trat_rf:.4f}")
print(f"  KS        Controle: {ks_ctrl_rf:.4f} | KS        Tratamento: {ks_trat_rf:.4f}")
print(f"  Uplift Médio (Teste): {uplift_rf.mean():.4f}")

In [ ]:
# Modelo 4 — Gradient Boosting (Ensemble Boosting — Avançado)
# Justificativa: Constrói árvores sequencialmente, onde cada nova árvore corrige os erros
# da anterior (residual fitting). Estado da arte em dados tabulares.
# Referência: Friedman, J. H. (2001). Greedy function approximation: a gradient boosting machine.
#             Annals of Statistics, 29(5), 1189-1232.

print("== MODELO 4: Gradient Boosting (Ensemble — Boosting) ==")

m_ctrl_gb, m_trat_gb = treinar_t_learner(
    GradientBoostingClassifier,
    {'n_estimators': 200, 'learning_rate': 0.05, 'max_depth': 4, 'random_state': 42},
    X_train, y_train, w_train)
uplift_gb = calcular_uplift(m_ctrl_gb, m_trat_gb, X_test)

auc_ctrl_gb  = roc_auc_score(y_test[w_test==0], m_ctrl_gb.predict_proba(X_test[w_test==0])[:,1])
auc_trat_gb  = roc_auc_score(y_test[w_test==1], m_trat_gb.predict_proba(X_test[w_test==1])[:,1])
acc_ctrl_gb  = accuracy_score(y_test[w_test==0], m_ctrl_gb.predict(X_test[w_test==0]))
acc_trat_gb  = accuracy_score(y_test[w_test==1], m_trat_gb.predict(X_test[w_test==1]))
f1_ctrl_gb   = f1_score(y_test[w_test==0], m_ctrl_gb.predict(X_test[w_test==0]))
f1_trat_gb   = f1_score(y_test[w_test==1], m_trat_gb.predict(X_test[w_test==1]))
prec_ctrl_gb = precision_score(y_test[w_test==0], m_ctrl_gb.predict(X_test[w_test==0]))
prec_trat_gb = precision_score(y_test[w_test==1], m_trat_gb.predict(X_test[w_test==1]))
rec_ctrl_gb  = recall_score(y_test[w_test==0], m_ctrl_gb.predict(X_test[w_test==0]))
rec_trat_gb  = recall_score(y_test[w_test==1], m_trat_gb.predict(X_test[w_test==1]))
ks_ctrl_gb   = calcular_ks(y_test[w_test==0].values, m_ctrl_gb.predict_proba(X_test[w_test==0])[:,1])
ks_trat_gb   = calcular_ks(y_test[w_test==1].values, m_trat_gb.predict_proba(X_test[w_test==1])[:,1])

print(f"  AUC       Controle: {auc_ctrl_gb:.4f} | AUC       Tratamento: {auc_trat_gb:.4f}")
print(f"  Accuracy  Controle: {acc_ctrl_gb:.4f} | Accuracy  Tratamento: {acc_trat_gb:.4f}")
print(f"  F1-Score  Controle: {f1_ctrl_gb:.4f} | F1-Score  Tratamento: {f1_trat_gb:.4f}")
print(f"  Precision Controle: {prec_ctrl_gb:.4f} | Precision Tratamento: {prec_trat_gb:.4f}")
print(f"  Recall    Controle: {rec_ctrl_gb:.4f} | Recall    Tratamento: {rec_trat_gb:.4f}")
print(f"  KS        Controle: {ks_ctrl_gb:.4f} | KS        Tratamento: {ks_trat_gb:.4f}")
print(f"  Uplift Médio (Teste): {uplift_gb.mean():.4f}")

In [ ]:
# Tabela Comparativa Global — Todos os Modelos e Métricas
# Justificativa: Centralizar métricas facilita a escolha do melhor modelo de forma objetiva.
# Usamos a média AUC como critério de seleção (equilibra performance nos dois grupos).
# Referência: Model Selection — James et al. (2021). Introduction to Statistical Learning. Springer.

tabela = pd.DataFrame({
    'Modelo': ['Naive Bayes', 'Reg. Logística', 'Random Forest', 'Gradient Boosting'],
    'AUC_Ctrl':  [auc_ctrl_nb,  auc_ctrl_lr,  auc_ctrl_rf,  auc_ctrl_gb],
    'AUC_Trat':  [auc_trat_nb,  auc_trat_lr,  auc_trat_rf,  auc_trat_gb],
    'Acc_Ctrl':  [acc_ctrl_nb,  acc_ctrl_lr,  acc_ctrl_rf,  acc_ctrl_gb],
    'Acc_Trat':  [acc_trat_nb,  acc_trat_lr,  acc_trat_rf,  acc_trat_gb],
    'F1_Ctrl':   [f1_ctrl_nb,   f1_ctrl_lr,   f1_ctrl_rf,   f1_ctrl_gb],
    'F1_Trat':   [f1_trat_nb,   f1_trat_lr,   f1_trat_rf,   f1_trat_gb],
    'Prec_Ctrl': [prec_ctrl_nb, prec_ctrl_lr, prec_ctrl_rf, prec_ctrl_gb],
    'Prec_Trat': [prec_trat_nb, prec_trat_lr, prec_trat_rf, prec_trat_gb],
    'Rec_Ctrl':  [rec_ctrl_nb,  rec_ctrl_lr,  rec_ctrl_rf,  rec_ctrl_gb],
    'Rec_Trat':  [rec_trat_nb,  rec_trat_lr,  rec_trat_rf,  rec_trat_gb],
    'KS_Ctrl':   [ks_ctrl_nb,   ks_ctrl_lr,   ks_ctrl_rf,   ks_ctrl_gb],
    'KS_Trat':   [ks_trat_nb,   ks_trat_lr,   ks_trat_rf,   ks_trat_gb],
    'Uplift_Med':[uplift_nb.mean(), uplift_lr.mean(), uplift_rf.mean(), uplift_gb.mean()]
}).round(4)

# Critério de seleção: Média AUC (Controle + Tratamento)
tabela['AUC_Media'] = ((tabela['AUC_Ctrl'] + tabela['AUC_Trat']) / 2).round(4)

print("📊 Tabela Comparativa Completa (Treino=70% | Teste=30%):")
print(tabela[['Modelo','AUC_Ctrl','AUC_Trat','AUC_Media','F1_Ctrl','F1_Trat',
              'Prec_Ctrl','Prec_Trat','Rec_Ctrl','Rec_Trat','KS_Ctrl','KS_Trat','Uplift_Med']].to_string(index=False))

In [ ]:
# Gráfico Comparativo — AUC por Modelo e Grupo
# Referência: Plotly Bar — https://plotly.com/python/bar-charts/

fig6 = go.Figure()
fig6.add_trace(go.Bar(name='AUC Controle',   x=tabela['Modelo'], y=tabela['AUC_Ctrl'],  marker_color='#636EFA'))
fig6.add_trace(go.Bar(name='AUC Tratamento', x=tabela['Modelo'], y=tabela['AUC_Trat'],  marker_color='#EF553B'))
fig6.add_trace(go.Bar(name='KS Controle',    x=tabela['Modelo'], y=tabela['KS_Ctrl'],   marker_color='#00CC96'))
fig6.add_trace(go.Bar(name='KS Tratamento',  x=tabela['Modelo'], y=tabela['KS_Trat'],   marker_color='#AB63FA'))
fig6.update_layout(title='📈 Comparativo: AUC-ROC e KS por Modelo e Grupo',
    barmode='group', template='plotly_white', yaxis_title='Score', yaxis_range=[0, 1.05])
fig6.show()

---
## 🔎 Etapa 4 — Interpretação, Avaliação e Seleção do Modelo

In [ ]:
# Seleção automática do melhor modelo pela AUC média
# Justificativa: A AUC média entre controle e tratamento garante que ambos sub-modelos sejam bons.
# Referência: Devriendt et al. (2018). A Literature Survey on Uplift Modeling. JMR.

idx_melhor = tabela['AUC_Media'].idxmax()
nome_melhor = tabela.loc[idx_melhor, 'Modelo']
print(f"�� Modelo Selecionado: {nome_melhor}")
print(f"   AUC Média: {tabela.loc[idx_melhor, 'AUC_Media']:.4f}")

# Mapeando os artefatos do modelo vencedor
mapa = {
    'Naive Bayes':       (m_ctrl_nb, m_trat_nb, uplift_nb),
    'Reg. Logística':    (m_ctrl_lr, m_trat_lr, uplift_lr),
    'Random Forest':     (m_ctrl_rf, m_trat_rf, uplift_rf),
    'Gradient Boosting': (m_ctrl_gb, m_trat_gb, uplift_gb),
}
mod_ctrl_final, mod_trat_final, uplift_final = mapa[nome_melhor]

In [ ]:
# Interpretabilidade Global — Permutation Importance no modelo vencedor
# Justificativa: Embaralha cada feature e mede queda no AUC. Feature mais importante = maior queda.
# Técnica modelo-agnóstica: funciona para qualquer algoritmo (NB, LR, RF, GB).
# Referência: sklearn.inspection.permutation_importance — https://scikit-learn.org/stable/modules/permutation_importance.html

resultado_pi = permutation_importance(
    mod_trat_final, X_test[w_test==1], y_test[w_test==1],
    n_repeats=15, random_state=42, scoring='roc_auc')

importancias = pd.DataFrame({
    'Variavel':   X.columns,
    'Importancia': resultado_pi.importances_mean,
    'Std':         resultado_pi.importances_std
}).sort_values('Importancia', ascending=True)

fig7 = px.bar(importancias, x='Importancia', y='Variavel', orientation='h',
    error_x='Std',
    title=f'🔍 Feature Importance Global — {nome_melhor} (Grupo Tratamento)',
    labels={'Importancia':'Impacto no AUC', 'Variavel':'Feature'},
    template='plotly_white', color='Importancia', color_continuous_scale='Blues')
fig7.show()

In [ ]:
# Distribuição do Uplift Score — Avaliação Local
# Justificativa: Boa discriminação = distribuição ampla e simétrica em torno do zero.
# Score muito concentrado num valor indica que o modelo não está diferenciando os clientes.
# Referência: Devriendt et al. (2018) — seção de distribuição de uplift scores.

fig8 = px.histogram(x=uplift_final, nbins=50,
    title='📊 Distribuição do Uplift Score Individual (Base de Teste)',
    labels={'x':'Uplift Score', 'y':'Volume de Clientes'},
    template='plotly_white', color_discrete_sequence=['#00CC96'])
fig8.add_vline(x=0, line_dash='dash', line_color='red', annotation_text='Neutro (0)')
fig8.show()

print(f"Estatísticas: Min={uplift_final.min():.4f} | Max={uplift_final.max():.4f} | Média={uplift_final.mean():.4f}")

---
## 🚀 Etapa 5 — Resultados Estratégicos (Storytelling)
**3 Segmentos baseados na Persuasion Matrix (Radcliffe, 2007):**
- 🟢 **Persuasíveis (Alto Uplift):** Foco total do budget
- 🟡 **Incertos (Médio Uplift):** Avaliar custo-benefício
- 🔴 **Não Contatar (Negativo):** Evitar desperdício ou efeito negativo


In [ ]:
# Score de Uplift para TODA a base e segmentação estratégica
# Justificativa: O Marketing precisa do score de todos os clientes ativos, não só do conjunto de teste.
# Referência: Production Scoring — Lakshmanan et al. (2020). Machine Learning Design Patterns. O'Reilly.

uplift_total = calcular_uplift(mod_ctrl_final, mod_trat_final, X)
df_resultado = df_base.copy()
df_resultado['uplift_score'] = uplift_total

def definir_segmento(score):
    if   score >= 0.05: return 'A — Alto Uplift (Persuasíveis)'
    elif score >  -0.05: return 'B — Médio Uplift (Incertos)'
    else:                return 'C — Baixo/Negativo (Não Contatar)'

df_resultado['segmento'] = df_resultado['uplift_score'].apply(definir_segmento)

resumo = df_resultado.groupby('segmento').agg(
    Volume=('id_cliente','count'),
    Uplift_Medio=('uplift_score','mean'),
    Renda_Media=('renda_mensal','mean'),
    Satisfacao_Media=('score_satisfacao','mean')
).round(3).reset_index()

print("📊 Resumo dos Segmentos Estratégicos:")
print(resumo.to_string(index=False))

In [ ]:
# Gráfico 1 — Pizza Estratégica de Alocação do Budget
# Referência: Plotly Pie — https://plotly.com/python/pie-charts/
cores_seg = ['#00CC96', '#FFA15A', '#EF553B']

fig9 = px.pie(resumo, values='Volume', names='segmento',
    title='🎯 Proposta Estratégica: Alocação da Base para Campanha',
    color_discrete_sequence=cores_seg, hole=0.45)
fig9.update_traces(textposition='outside', textinfo='percent+label')
fig9.update_layout(template='plotly_white')
fig9.show()

In [ ]:
# Gráfico 2 — Boxplot de Uplift por Segmento (Validação da Separação)
fig10 = px.box(df_resultado, x='segmento', y='uplift_score', color='segmento',
    title='⚖️ Validação dos Segmentos: Uplift Score por Grupo Estratégico',
    labels={'segmento':'Segmento', 'uplift_score':'Uplift Score'},
    color_discrete_sequence=cores_seg, template='plotly_white')
fig10.add_hline(y=0, line_dash='dash', line_color='black', annotation_text='Neutro (0)')
fig10.update_layout(showlegend=False)
fig10.show()

In [ ]:
# Gráfico 3 — Perfil de Renda e Satisfação por Segmento
fig11 = make_subplots(rows=1, cols=2,
    subplot_titles=['Renda Média (R$)', 'Satisfação Média'])
for i, col in enumerate(['Renda_Media', 'Satisfacao_Media']):
    fig11.add_trace(go.Bar(
        x=resumo['segmento'], y=resumo[col],
        marker_color=cores_seg, text=resumo[col].round(1),
        textposition='auto', showlegend=False), row=1, col=i+1)
fig11.update_layout(title_text='👥 Perfil dos Segmentos de Marketing',
    template='plotly_white', height=400)
fig11.show()

In [ ]:
# Simulação de ROI — Economia com Uplift Model vs. Campanha Universal
# Justificativa: Quantificar em R$ o valor gerado pelo modelo para justificar o investimento.
# Referência: Anderson & Simester (2011). Smart Business Experiments. Harvard Business Review.

CUSTO_ACAO     = 65.00  # R$15 contato + R$50 incentivo médio
total          = len(df_resultado)
vol_persuasiv  = len(df_resultado[df_resultado['segmento']=='A — Alto Uplift (Persuasíveis)'])

custo_universal = total        * CUSTO_ACAO
custo_uplift    = vol_persuasiv * CUSTO_ACAO
economia        = custo_universal - custo_uplift

print("=" * 60)
print("  💰 ROI SIMULADO — UPLIFT MODEL vs. CAMPANHA UNIVERSAL")
print("=" * 60)
print(f"  Total de Clientes:                    {total:>8,}")
print(f"  Custo por Ação (contato + incentivo): R$ {CUSTO_ACAO:>6,.2f}")
print(f"")
print(f"  [Atual] Envia para TODOS:             R$ {custo_universal:>10,.2f}")
print(f"  [Novo]  Envia só Persuasíveis:        R$ {custo_uplift:>10,.2f}")
print(f"")
print(f"  🟢 ECONOMIA ESTIMADA:                 R$ {economia:>10,.2f}")
print(f"  🟢 REDUÇÃO DE CUSTO:                   {economia/custo_universal*100:.1f}%")
print("=" * 60)

---
## 🏁 Conclusão Executiva

Com base no modelo **T-Learner com Gradient Boosting**, identificamos 3 segmentos estratégicos:

| Segmento | Ação | Justificativa |
|---|---|---|
| 🟢 **Persuasíveis** | **Enviar campanha** | Alto Uplift — campanha gera retenção incremental real |
| 🟡 **Incertos** | **Comunicação leve** | Uplift neutro — avaliar custo x benefício por canal mais barato |
| 🔴 **Não Contatar** | **Suspender contato** | Uplift negativo — campanha pode irritar ou è desperdício de verba |

**Resultado:** Substituímos o "marketing de batedeira" pela precisão cirúrgica do Uplift Modeling, gerando economia significativa e maior ROI por real investido em retenção.

---
*Todas as referências estão declaradas na célula de cabeçalho e em cada célula de código.*
